In [ ]:
import pandas as pd
import re
import geopandas as gpd
import matplotlib.pyplot as plt
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

In [ ]:
COUNTRIES = {
    "germany": "Germany", "deutschland": "Germany",
    "usa": "USA", "united states": "USA",
    "netherlands": "Netherlands", "czech": "Czech Republic",
    "austria": "Austria", "switzerland": "Switzerland",
    "france": "France", "united kingdom": "United Kingdom", "uk": "United Kingdom",
}
TLD = {"de": "Germany", "nl": "Netherlands", "cz": "Czech Republic",
       "at": "Austria", "ch": "Switzerland", "fr": "France", "uk": "United Kingdom"}
AFFILIATION_CITY = {
    r"göttingen|gottingen|gwdg|\bsub\b|\bumg\b|indiscale": "Göttingen",
    r"jülich|julich": "Jülich",
    r"potsdam|\bpik\b|plant physiology": "Potsdam",
    r"heidelberg|\biwr\b|\bssc\b": "Heidelberg",
    r"dkrz|bernhard.nocht|bnitm": "Hamburg",
    r"braunschweig|thünen": "Braunschweig",
    r"zb med|\bdlr\b": "Cologne",
    r"bonn": "Bonn",
    r"dresden|hzdr|rossendorf|mpi-cbg|molecular cell biology": "Dresden",
    r"augsburg": "Augsburg",
    r"\bufz\b|leipzig": "Leipzig",
    r"marburg": "Marburg",
    r"alfred.wegener": "Bremerhaven",
    r"oldenburg": "Oldenburg",
    r"jena|geoanthropology": "Jena",
    r"berlin|weierstrass|\bwzb\b": "Berlin",
    r"\bkit\b|karlsruhe": "Karlsruhe",
    r"magdeburg|dynamik komplexer": "Magdeburg",
    r"\btib\b|hannover": "Hannover",
    r"rwth|aachen": "Aachen",
    r"hereon": "Geesthacht",
    r"kiel": "Kiel",
    r"münster|munster": "Münster",
    r"\bfau\b|erlangen": "Erlangen",
    r"max planck computing|mpcdf": "Garching",
}
# hands-on sessions per day, rated 0 (least) to 3 (most preferred)
DAY1_SESSIONS = [
    "Building Software by Talking to AI -and Making It Run Faster",
    "Gemini as a Coding Assistant and Debugging Collaborator Using AMERH Framework",
    "Quality Assurance in the Age of AI",
    "MCP, Skills and RAG for different applications",
]
DAY2_SESSIONS = [
    "AI Declaration Format",
    "Vibe Coding: A Useful Science Tool",
    "Semantic Search using LLMs plus Hands-on agentic coding for research software development: experiences and challenges",
    "From Code Generation to Control: Engineering Reliable Software with AI",
]

In [ ]:
def infer_country(row):
    address = row["Address"] if pd.notna(row["Address"]) else ""
    for key, country in COUNTRIES.items():
        if re.search(rf"\b{key}\b", address.lower()):
            return country
    tld = row["Email Address"].rsplit(".", 1)[-1].lower()
    if tld in TLD:
        return TLD[tld]
    if re.search(r"\b\d{5}\b", address):
        return "Germany"
    return None

def infer_city(row):
    if pd.notna(row["Affiliation"]):
        for pattern, city in AFFILIATION_CITY.items():
            if re.search(pattern, row["Affiliation"].lower()):
                return city
    # fallback: city name following a 5-digit postcode in the address
    if pd.notna(row["Address"]):
        match = re.search(r"\b\d{5}\s+([A-ZÄÖÜ][\w\-äöüß]+)", row["Address"])
        if match:
            return match.group(1)
    return None

def session_votes(sessions):
    # every 3 counts as a vote; participants without any 3 (or no answer) abstain
    threes = df[sessions] == 3
    votes = threes.sum().sort_values(ascending=False)
    votes["abstain (no 3)"] = (threes.sum(axis=1) == 0).sum()
    multiple = (threes.sum(axis=1) > 1).sum()
    print(f"{len(df)} participants, {multiple} gave more than one 3, so the votes sum to {votes.sum()}")
    return votes.to_frame("votes")

In [ ]:
df = pd.read_csv("registrations.csv")

In [ ]:
df.head()

In [ ]:
# show which country participants are from
df["Country"] = df.apply(infer_country, axis=1)
df.head(5)

In [ ]:
df["Country"].value_counts(dropna=False)

In [ ]:
# show which city participants are from
df["City"] = df.apply(infer_city, axis=1)
# no affiliation and no usable address: set manually
df.loc[27, "City"] = "Regensburg"  # ur.de email
df.loc[14, "City"] = "Bad Buchau"  # postcode after city name
df[["Affiliation", "City"]].head(5)

In [ ]:
df.loc[df["Country"] == "Germany", "City"].value_counts(dropna=False)
# df.loc[df["Country"] == "Germany", "City"].value_counts(dropna=False).sum()

In [ ]:
# geocode city names only (no participant addresses are sent)
geocode = RateLimiter(Nominatim(user_agent="rse-workshop-map").geocode, min_delay_seconds=1)

cities = df[df["City"].notna() & (df["Country"] == "Germany")].groupby("City").size().reset_index(name="n")
locations = cities["City"].apply(lambda c: geocode(f"{c}, Germany"))
cities["lat"] = locations.apply(lambda l: l.latitude)
cities["lon"] = locations.apply(lambda l: l.longitude)
cities

In [ ]:
world = gpd.read_file("https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip")
germany = world[world["ADM0_A3"] == "DEU"]
# label position for cities that would overlap their neighbours (default: right)
LABEL_POSITION = {
    "Aachen": "left", "Jülich": "above", "Bremerhaven": "left", "Potsdam": "left",
    "Hannover": "above", "Braunschweig": "above", "Augsburg": "left",
}

# draw large circles first so small ones stay visible on top
plot_cities = cities.sort_values("n", ascending=False)
# circle area grows with participant count, minimum size so the number fits
sizes = 80 + 25 * plot_cities["n"]

fig, ax = plt.subplots(figsize=(6, 8))
germany.plot(ax=ax, color="#eeeeee", edgecolor="#666666", linewidth=0.8)
for (_, row), size in zip(plot_cities.iterrows(), sizes):
    ax.scatter(row["lon"], row["lat"], s=size, color="#2a6f97", edgecolor="white", linewidth=1)
    # count inside the circle, city name next to it
    ax.text(row["lon"], row["lat"], row["n"], color="white", fontsize=7, ha="center", va="center")
    offset = size**0.5 / 2 + 2
    xytext, ha, va = {
        "right": ((offset, 0), "left", "center"),
        "left": ((-offset, 0), "right", "center"),
        "above": ((0, offset), "center", "bottom"),
        "below": ((0, -offset), "center", "top"),
    }[LABEL_POSITION.get(row["City"], "right")]
    ax.annotate(row["City"], (row["lon"], row["lat"]), xytext=xytext, textcoords="offset points",
                fontsize=7, ha=ha, va=va, color="#333333")
ax.set_axis_off()

fig.savefig("participants_map.png", bbox_inches="tight")
plt.show()

In [ ]:
df["How do you judge your experience level?"].value_counts(dropna=False)

In [ ]:
# find the entries that showed NaN
df[df["How do you judge your experience level?"].isna()][["Name", "Affiliation"]]

In [ ]:
# find the entries that showed "Expert"
df[df["How do you judge your experience level?"] == "Expert"][["Name", "Affiliation"]]

In [ ]:
# day 1
session_votes(DAY1_SESSIONS)

In [ ]:
# day 2
session_votes(DAY2_SESSIONS)